# Ablation 5: Statistical Significance via Multiple Seeds

**Goal**: Prove that the CoT-augmented improvement over Direct LoRA is consistent and not a lucky seed result.

**What changes per seed**: Only `torch.manual_seed()` — this affects:
- LoRA adapter weight initialization (random)
- Training data shuffle order per epoch
- Dropout mask during training

**What stays fixed**:
- Training images (same JSONL files already cached)
- CoT descriptions (already generated and cached)
- Prompts (identical)
- Hyperparameters (LR=2e-5, epochs=40, r=16, alpha=32)
- Test set (same images per task)
- Evaluation (temp=0.1, same parsing logic)

**Seeds**: [42, 123, 456, 789, 1024]

**Tasks**: Granulometry, Steel Surface, UHCS, Weld Defects (4 tasks × 2 approaches × 5 seeds = 40 runs)

## Setup & Imports

In [ ]:
import os, sys, json, re, gc, time, random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from collections import defaultdict

# Paths
SCRIPT_DIR = Path(os.getcwd())
REPO_ROOT = SCRIPT_DIR.parent.parent
TASK4_DIR = REPO_ROOT / 'task4-fine-tuning'

# Seeds (42 is excluded — those are our existing baseline results)
SEEDS = [123, 456, 789, 1024]

# Existing seed-42 results (from task4-fine-tuning)
SEED42_RESULTS = {
    'granulometry': {'direct': 71.3, 'augmented': 79.6},
    'steel_surface': {'direct': 63.1, 'augmented': 66.7},
    'uhcs': {'direct': 67.5, 'augmented': 68.4},
    'weld': {'direct': 73.3, 'augmented': 75.8},
}

# Model
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# LoRA config (identical across all tasks)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# Training
EPOCHS = 40
LR = 2e-5
GRAD_ACCUM = 4

# Evaluation
EVAL_TEMPERATURE = 0.1

# Results directory
RESULTS_DIR = SCRIPT_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'TASK4_DIR: {TASK4_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

## Utility Functions

In [ ]:
def set_all_seeds(seed):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def parse_json_response(raw):
    """Parse JSON from model response, handling markdown fences and CoT prefixes."""
    if not raw:
        return None
    raw = raw.replace('<', '').replace('>', '')
    cleaned = re.sub(r'```json\s*', '', raw)
    cleaned = re.sub(r'```\s*', '', cleaned).strip()
    # Try direct parse
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except (json.JSONDecodeError, ValueError):
        pass
    # Find last JSON object in the response (handles CoT + JSON)
    matches = list(re.finditer(r'\{[^{}]*\}', cleaned))
    if matches:
        try:
            return json.loads(matches[-1].group())
        except (json.JSONDecodeError, ValueError):
            pass
    return None


print('Utility functions ready.')

## Dataset Class (Universal for all tasks)

In [ ]:
class LoRADataset(torch.utils.data.Dataset):
    """Universal dataset for all 4 tasks — reads JSONL training data."""

    def __init__(self, jsonl_path, processor):
        with open(jsonl_path) as f:
            self.data = [json.loads(line) for line in f]
        self.processor = processor
        self.base_dir = os.path.dirname(jsonl_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        msgs = entry['messages']

        # Extract image path and text from user message
        img_path = None
        user_text = ''
        for content in msgs[0]['content']:
            if content['type'] == 'image':
                img_path = content['image']
            elif content['type'] == 'text':
                user_text = content['text']

        # Resolve relative image path
        if img_path and not os.path.isabs(img_path):
            img_path = os.path.normpath(os.path.join(self.base_dir, img_path))

        # Get assistant response
        assistant_text = msgs[1]['content']
        if not isinstance(assistant_text, str):
            assistant_text = json.dumps(assistant_text)

        # Load image
        image = Image.open(img_path).convert('RGB') if img_path else None

        # Build chat and tokenize
        chat = [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': user_text}
            ]},
            {'role': 'assistant', 'content': [
                {'type': 'text', 'text': assistant_text}
            ]}
        ]
        text = self.processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
        inputs = self.processor(text=[text], images=[image], return_tensors='pt', padding=True)

        input_ids = inputs['input_ids'].squeeze(0)
        labels = input_ids.clone()

        # Mask everything except assistant tokens
        ast_tokens = self.processor.tokenizer.encode(assistant_text, add_special_tokens=False)
        if len(ast_tokens) < len(labels):
            labels[:-len(ast_tokens)] = -100

        if image:
            image.close()

        return {
            'input_ids': input_ids,
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': labels,
            'pixel_values': inputs.get('pixel_values', None),
            'image_grid_thw': inputs.get('image_grid_thw', None),
        }


print('Dataset class ready.')

## Training Function

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import get_cosine_schedule_with_warmup


def train_lora(base_model, processor, jsonl_path, output_dir, seed):
    """Train a LoRA adapter with the given seed."""
    # Set seed BEFORE LoRA initialization
    set_all_seeds(seed)

    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, task_type='CAUSAL_LM', bias='none'
    )
    model = get_peft_model(base_model, lora_config)
    model.gradient_checkpointing_enable()
    model.print_trainable_parameters()

    dataset = LoRADataset(jsonl_path, processor)
    print(f'  Training: {len(dataset)} examples, {EPOCHS} epochs, lr={LR}, seed={seed}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = max(len(dataset) * EPOCHS // GRAD_ACCUM, 1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)

    model.train()
    t_start = time.time()

    for epoch in range(EPOCHS):
        # Iterate sequentially (same as original notebooks — no shuffling)
        epoch_loss = 0
        n_ok = 0
        for step in range(len(dataset)):
            try:
                batch = dataset[step]
                ids = batch['input_ids'].unsqueeze(0).to(model.device)
                mask = batch['attention_mask'].unsqueeze(0).to(model.device)
                lab = batch['labels'].unsqueeze(0).to(model.device)

                kw = {'input_ids': ids, 'attention_mask': mask, 'labels': lab}
                if batch.get('pixel_values') is not None:
                    kw['pixel_values'] = batch['pixel_values'].to(model.device)
                if batch.get('image_grid_thw') is not None:
                    kw['image_grid_thw'] = batch['image_grid_thw'].to(model.device)

                out = model(**kw)
                loss = out.loss / GRAD_ACCUM
                loss.backward()
                epoch_loss += loss.item() * GRAD_ACCUM
                n_ok += 1

                if (step + 1) % GRAD_ACCUM == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                del ids, mask, lab, out, loss
                torch.cuda.empty_cache()
            except Exception as e:
                print(f'    Skip idx {i} (epoch {epoch+1}): {e}')
                optimizer.zero_grad()
                torch.cuda.empty_cache()

        avg_loss = epoch_loss / max(n_ok, 1)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            elapsed = time.time() - t_start
            print(f'    Epoch {epoch+1}/{EPOCHS} — loss: {avg_loss:.4f} — lr: {scheduler.get_last_lr()[0]:.2e} — {elapsed:.0f}s')

    # Save adapter
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    print(f'  Adapter saved to {output_dir}')

    # Cleanup
    model.unload()
    del model, optimizer, scheduler, dataset
    gc.collect()
    torch.cuda.empty_cache()


print('Training function ready.')

## Evaluation Functions

In [ ]:
def evaluate_granulometry(model, processor, task_config):
    """Evaluate granulometry: metric is 'both correct' (size AND grading)."""
    ORIGINAL_GSD = 8.0
    MAX_DIM = 800
    test_manifest_path = str(task_config['test_manifest'])
    test_dir = str(task_config['test_dir'])

    with open(test_manifest_path) as f:
        manifest = json.load(f)

    model.eval()
    correct = 0
    total = 0

    for i, entry in enumerate(manifest):
        img_path = os.path.join(test_dir, entry['image'])
        if not os.path.exists(img_path):
            continue

        image = Image.open(img_path).convert('RGB')
        scale = min(MAX_DIM / max(image.size), 1.0)
        gsd = ORIGINAL_GSD * scale

        prompt = f"""Classify this concrete aggregate photograph.
Ground sampling distance (GSD) = {gsd:.1f} px/mm.
At this GSD: 8mm stone \u2248 {8*gsd:.0f}px, 16mm \u2248 {16*gsd:.0f}px, 32mm \u2248 {32*gsd:.0f}px.

Classification axes:
1. MAX PARTICLE SIZE: estimate the largest stone's width in pixels, divide by GSD, round to 8, 16, or 32 mm.
2. GRADING (DIN 1045 standard \u2014 describes size DISTRIBUTION, not absolute size):
   - COARSE (A): particles concentrated near max size. Gaps between stones are EMPTY. Uniform, single-layer texture.
   - MEDIUM (B): balanced mix. Gaps PARTIALLY filled by smaller particles.
   - FINE (C): wide size range. Gaps COMPLETELY filled with small particles. Dense, packed texture.

Respond with JSON: {{\"max_particle_size_mm\": <8, 16, or 32>, \"grading\": \"<coarse, medium, or fine>\"}}"""

        msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)

        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=128, temperature=EVAL_TEMPERATURE, do_sample=True)
        raw = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

        del inputs, ids
        image.close()
        torch.cuda.empty_cache()

        parsed = parse_json_response(raw)
        gt_size = entry['max_particle_size_mm']
        gt_grading = entry['grading']

        if parsed:
            pred_size = parsed.get('max_particle_size_mm')
            if isinstance(pred_size, str):
                pred_size = int(pred_size) if pred_size.isdigit() else None
            pred_grading = parsed.get('grading', '').lower().strip()
            if pred_size == gt_size and pred_grading == gt_grading:
                correct += 1
        total += 1

        if (i + 1) % 20 == 0:
            print(f'    [{i+1}/{len(manifest)}] Both correct: {correct}/{total} ({correct/total*100:.1f}%)')

    accuracy = (correct / total * 100) if total > 0 else 0.0
    print(f'  Result: {correct}/{total} ({accuracy:.1f}%) both correct')
    return accuracy


def evaluate_classification(model, processor, task_config, task_name):
    """Evaluate steel/weld/uhcs: metric is single-class accuracy."""

    # Load test images
    if task_name == 'uhcs':
        with open(str(task_config['test_manifest'])) as f:
            manifest = json.load(f)
        # UHCS manifest has image path, class, magnification
    elif task_name == 'weld':
        # Weld: sample 60 per class with fixed seed
        import random as _r
        _r.seed(42)
        manifest = []
        test_dir = str(task_config['test_dir'])
        for cls in task_config['classes']:
            cls_dir = os.path.join(test_dir, cls)
            images = sorted([f for f in os.listdir(cls_dir) if f.endswith('.png')])
            if len(images) > 60:
                images = _r.sample(images, 60)
            for img in images:
                manifest.append({'image': os.path.join(cls_dir, img), 'class': cls})
        _r.shuffle(manifest)
    else:
        # Steel: all images from validation dirs
        manifest = []
        test_dir = str(task_config['test_dir'])
        for cls in task_config['classes']:
            cls_dir = os.path.join(test_dir, cls)
            for f in sorted(os.listdir(cls_dir)):
                if f.lower().endswith(('.jpg', '.png', '.bmp')):
                    manifest.append({'image': os.path.join(cls_dir, f), 'class': cls})

    # Get eval prompt from first direct JSONL entry
    with open(str(task_config['direct_jsonl'])) as f:
        first = json.loads(f.readline())
    static_prompt = next((c['text'] for c in first['messages'][0]['content'] if c['type'] == 'text'), '')

    # Determine JSON key
    key = 'primary_microconstituent' if task_name == 'uhcs' else 'defect_class'

    model.eval()
    correct = 0
    total = 0

    for i, entry in enumerate(manifest):
        # Get image path
        if task_name == 'uhcs':
            img_path = entry.get('image_path', entry.get('image', ''))
        else:
            img_path = entry['image']

        if not os.path.exists(img_path):
            continue

        image = Image.open(img_path).convert('RGB')

        # Build prompt — UHCS needs magnification per image
        if task_name == 'uhcs':
            mag = entry.get('magnification', 'unknown')
            prompt = f"""Classify this ultra-high carbon steel (UHCS) micrograph.

This is an optical/SEM micrograph at approximately {mag} magnification showing the microstructure of UHCS after heat treatment.

Possible microconstituent classes:
1. spheroidite: Scattered dark round/oval cementite particles on a light ferrite matrix. The particles are isolated, roughly spherical, and uniformly distributed. Looks like \"polka dots.\" This forms from prolonged annealing below the eutectoid temperature.
2. network: Dark continuous lines forming a connected web/mesh pattern. These are cementite films along prior austenite grain boundaries. The lines outline polygonal grain shapes. Forms during slow cooling from above A1.
3. spheroidite+widmanstatten: A mix of round spheroidized particles AND straight elongated needle/plate-like features growing inward from grain boundaries. You see both \"dots\" and \"needles\" in the same image. Indicates partial spheroidization of Widmanstatten cementite.
4. pearlite+spheroidite: Regions showing fingerprint-like lamellar striations (pearlite) alongside areas with scattered round particles (spheroidite). Two distinct textures coexist. Indicates incomplete spheroidization of pearlite.
5. pearlite: Fine parallel alternating dark/light lamellae creating a fingerprint or wood-grain pattern. Very regular, closely-spaced striations. Requires high magnification to resolve individual lamellae.

Respond with ONLY a JSON object:
{{\"primary_microconstituent\": \"<spheroidite|network|spheroidite+widmanstatten|pearlite+spheroidite|pearlite>\"}}"""
        else:
            prompt = static_prompt

        msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)

        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=256, temperature=EVAL_TEMPERATURE, do_sample=True)
        raw = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

        del inputs, ids
        image.close()
        torch.cuda.empty_cache()

        parsed = parse_json_response(raw)
        gt_class = entry['class']

        if parsed:
            pred = parsed.get(key, '').lower().strip()
            if pred == gt_class.lower().strip():
                correct += 1
        total += 1

        if (i + 1) % 60 == 0:
            print(f'    [{i+1}/{len(manifest)}] Acc: {correct}/{total} ({correct/total*100:.1f}%)')

    accuracy = (correct / total * 100) if total > 0 else 0.0
    print(f'  Result: {correct}/{total} ({accuracy:.1f}%)')
    return accuracy


print('Evaluation functions ready.')

## Train + Evaluate Pipeline

In [ ]:
def train_and_evaluate(task_name, task_config, approach, seed):
    """
    Full pipeline: load model -> train LoRA -> evaluate -> cleanup.
    Returns accuracy (float, percentage).
    """
    jsonl_path = str(task_config['direct_jsonl'] if approach == 'direct' else task_config['augmented_jsonl'])

    print(f'  Loading training data from: {jsonl_path}')
    with open(jsonl_path) as f:
        n_examples = sum(1 for _ in f)
    print(f'  Training examples: {n_examples}')

    # Load base model
    print(f'  Loading {MODEL_ID}...')
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
    )
    base_model.enable_input_require_grads()
    print(f'  Model loaded in {time.time()-t0:.1f}s')

    # Train
    adapter_dir = str(RESULTS_DIR / f'{task_name}_{approach}_seed{seed}_adapter')
    train_lora(base_model, processor, jsonl_path, adapter_dir, seed)

    # Load adapter for evaluation
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()

    # Evaluate
    if task_name == 'granulometry':
        accuracy = evaluate_granulometry(model, processor, task_config)
    else:
        accuracy = evaluate_classification(model, processor, task_config, task_name)

    # Cleanup
    del model, base_model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  GPU memory freed.')

    return accuracy


print('Pipeline ready.')

---
## GRANULOMETRY

- Direct: 18 training examples, 40 epochs
- Augmented: 72 training examples, 40 epochs
- Test: 108 images, metric = both correct

In [ ]:
GRANULOMETRY_CONFIG = {
    'direct_jsonl': TASK4_DIR / 'granulometry' / 'training_data_direct.jsonl',
    'augmented_jsonl': TASK4_DIR / 'granulometry' / 'training_data_augmented.jsonl',
    'test_manifest': REPO_ROOT / 'datasets' / 'granulometry' / 'test_manifest.json',
    'test_dir': REPO_ROOT / 'datasets' / 'granulometry' / 'test',
}

granulometry_results = {'direct': {}, 'augmented': {}}

for approach in ['direct', 'augmented']:
    for seed in SEEDS:
        run_id = f'granulometry_{approach}_seed{seed}'
        result_file = RESULTS_DIR / f'{run_id}.json'

        # Skip if already done
        if result_file.exists():
            with open(result_file) as f:
                existing = json.load(f)
            granulometry_results[approach][seed] = existing['accuracy']
            print(f'[SKIP] {run_id}: {existing["accuracy"]:.1f}%')
            continue

        print(f'\n{"="*60}')
        print(f'  GRANULOMETRY | {approach} | seed={seed}')
        print(f'{"="*60}')

        t_start = time.time()
        accuracy = train_and_evaluate('granulometry', GRANULOMETRY_CONFIG, approach, seed)
        elapsed = time.time() - t_start

        granulometry_results[approach][seed] = accuracy

        with open(result_file, 'w') as f:
            json.dump({'task': 'granulometry', 'approach': approach, 'seed': seed,
                       'accuracy': accuracy, 'elapsed_min': round(elapsed/60, 1)}, f, indent=2)

        print(f'  -> {accuracy:.1f}% (took {elapsed/60:.1f} min)')

In [ ]:
# Granulometry Summary
print('\nGRANULOMETRY RESULTS:')
for approach in ['direct', 'augmented']:
    accs = [SEED42_RESULTS['granulometry'][approach]] + [granulometry_results[approach].get(s) for s in SEEDS]
    accs = [a for a in accs if a is not None]
    if accs:
        print(f'  {approach:12s}: {np.mean(accs):.1f} \u00b1 {np.std(accs, ddof=1):.1f}%  seeds={[f"{a:.1f}" for a in accs]}')

d = [SEED42_RESULTS['granulometry']['direct']] + [granulometry_results['direct'].get(s) for s in SEEDS]
a = [SEED42_RESULTS['granulometry']['augmented']] + [granulometry_results['augmented'].get(s) for s in SEEDS]
paired = [(av - dv) for dv, av in zip(d, a) if dv is not None and av is not None]
if paired:
    print(f'  DELTA       : +{np.mean(paired):.1f} \u00b1 {np.std(paired, ddof=1):.1f}pp')

---
## STEEL SURFACE

- Direct: 30 training examples, 40 epochs
- Augmented: 120 training examples, 40 epochs
- Test: 360 images (60 per class), metric = accuracy

In [ ]:
STEEL_CONFIG = {
    'direct_jsonl': TASK4_DIR / 'steel-surface' / 'training_data_direct.jsonl',
    'augmented_jsonl': TASK4_DIR / 'steel-surface' / 'training_data_augmented.jsonl',
    'test_dir': REPO_ROOT / 'datasets' / 'neu-cls' / 'NEU-DET' / 'validation' / 'images',
    'classes': ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches'],
}

steel_results = {'direct': {}, 'augmented': {}}

for approach in ['direct', 'augmented']:
    for seed in SEEDS:
        run_id = f'steel_surface_{approach}_seed{seed}'
        result_file = RESULTS_DIR / f'{run_id}.json'

        if result_file.exists():
            with open(result_file) as f:
                existing = json.load(f)
            steel_results[approach][seed] = existing['accuracy']
            print(f'[SKIP] {run_id}: {existing["accuracy"]:.1f}%')
            continue

        print(f'\n{"="*60}')
        print(f'  STEEL SURFACE | {approach} | seed={seed}')
        print(f'{"="*60}')

        t_start = time.time()
        accuracy = train_and_evaluate('steel_surface', STEEL_CONFIG, approach, seed)
        elapsed = time.time() - t_start

        steel_results[approach][seed] = accuracy

        with open(result_file, 'w') as f:
            json.dump({'task': 'steel_surface', 'approach': approach, 'seed': seed,
                       'accuracy': accuracy, 'elapsed_min': round(elapsed/60, 1)}, f, indent=2)

        print(f'  -> {accuracy:.1f}% (took {elapsed/60:.1f} min)')

In [ ]:
# Steel Surface Summary
print('\nSTEEL SURFACE RESULTS:')
for approach in ['direct', 'augmented']:
    accs = [SEED42_RESULTS['steel_surface'][approach]] + [steel_results[approach].get(s) for s in SEEDS]
    accs = [a for a in accs if a is not None]
    if accs:
        print(f'  {approach:12s}: {np.mean(accs):.1f} \u00b1 {np.std(accs, ddof=1):.1f}%  seeds={[f"{a:.1f}" for a in accs]}')

d = [SEED42_RESULTS['steel_surface']['direct']] + [steel_results['direct'].get(s) for s in SEEDS]
a = [SEED42_RESULTS['steel_surface']['augmented']] + [steel_results['augmented'].get(s) for s in SEEDS]
paired = [(av - dv) for dv, av in zip(d, a) if dv is not None and av is not None]
if paired:
    print(f'  DELTA       : +{np.mean(paired):.1f} \u00b1 {np.std(paired, ddof=1):.1f}pp')

---
## UHCS MICROSTRUCTURE

- Direct: 30 training examples, 40 epochs
- Augmented: 120 training examples, 40 epochs
- Test: ~117-120 images, metric = accuracy

In [ ]:
UHCS_CONFIG = {
    'direct_jsonl': TASK4_DIR / 'uhcs-microstructure' / 'training_data_direct.jsonl',
    'augmented_jsonl': TASK4_DIR / 'uhcs-microstructure' / 'training_data_augmented.jsonl',
    'test_manifest': REPO_ROOT / 'datasets' / 'uh-carbon-steel' / 'test_manifest.json',
}

uhcs_results = {'direct': {}, 'augmented': {}}

for approach in ['direct', 'augmented']:
    for seed in SEEDS:
        run_id = f'uhcs_{approach}_seed{seed}'
        result_file = RESULTS_DIR / f'{run_id}.json'

        if result_file.exists():
            with open(result_file) as f:
                existing = json.load(f)
            uhcs_results[approach][seed] = existing['accuracy']
            print(f'[SKIP] {run_id}: {existing["accuracy"]:.1f}%')
            continue

        print(f'\n{"="*60}')
        print(f'  UHCS | {approach} | seed={seed}')
        print(f'{"="*60}')

        t_start = time.time()
        accuracy = train_and_evaluate('uhcs', UHCS_CONFIG, approach, seed)
        elapsed = time.time() - t_start

        uhcs_results[approach][seed] = accuracy

        with open(result_file, 'w') as f:
            json.dump({'task': 'uhcs', 'approach': approach, 'seed': seed,
                       'accuracy': accuracy, 'elapsed_min': round(elapsed/60, 1)}, f, indent=2)

        print(f'  -> {accuracy:.1f}% (took {elapsed/60:.1f} min)')

In [ ]:
# UHCS Summary
print('\nUHCS RESULTS:')
for approach in ['direct', 'augmented']:
    accs = [SEED42_RESULTS['uhcs'][approach]] + [uhcs_results[approach].get(s) for s in SEEDS]
    accs = [a for a in accs if a is not None]
    if accs:
        print(f'  {approach:12s}: {np.mean(accs):.1f} \u00b1 {np.std(accs, ddof=1):.1f}%  seeds={[f"{a:.1f}" for a in accs]}')

d = [SEED42_RESULTS['uhcs']['direct']] + [uhcs_results['direct'].get(s) for s in SEEDS]
a = [SEED42_RESULTS['uhcs']['augmented']] + [uhcs_results['augmented'].get(s) for s in SEEDS]
paired = [(av - dv) for dv, av in zip(d, a) if dv is not None and av is not None]
if paired:
    print(f'  DELTA       : +{np.mean(paired):.1f} \u00b1 {np.std(paired, ddof=1):.1f}pp')

---
## WELD DEFECTS

- Direct: 24 training examples, 40 epochs
- Augmented: 96 training examples, 40 epochs
- Test: 240 images (60 per class), metric = accuracy

In [ ]:
WELD_CONFIG = {
    'direct_jsonl': TASK4_DIR / 'riawelc-weld' / 'training_data_direct.jsonl',
    'augmented_jsonl': TASK4_DIR / 'riawelc-weld' / 'training_data_augmented.jsonl',
    'test_dir': REPO_ROOT / 'datasets' / 'riawelc' / 'testing',
    'classes': ['lack_of_penetration', 'porosity', 'cracks', 'no_defect'],
}

weld_results = {'direct': {}, 'augmented': {}}

for approach in ['direct', 'augmented']:
    for seed in SEEDS:
        run_id = f'weld_{approach}_seed{seed}'
        result_file = RESULTS_DIR / f'{run_id}.json'

        if result_file.exists():
            with open(result_file) as f:
                existing = json.load(f)
            weld_results[approach][seed] = existing['accuracy']
            print(f'[SKIP] {run_id}: {existing["accuracy"]:.1f}%')
            continue

        print(f'\n{"="*60}')
        print(f'  WELD DEFECTS | {approach} | seed={seed}')
        print(f'{"="*60}')

        t_start = time.time()
        accuracy = train_and_evaluate('weld', WELD_CONFIG, approach, seed)
        elapsed = time.time() - t_start

        weld_results[approach][seed] = accuracy

        with open(result_file, 'w') as f:
            json.dump({'task': 'weld', 'approach': approach, 'seed': seed,
                       'accuracy': accuracy, 'elapsed_min': round(elapsed/60, 1)}, f, indent=2)

        print(f'  -> {accuracy:.1f}% (took {elapsed/60:.1f} min)')

In [ ]:
# Weld Summary
print('\nWELD DEFECTS RESULTS:')
for approach in ['direct', 'augmented']:
    accs = [SEED42_RESULTS['weld'][approach]] + [weld_results[approach].get(s) for s in SEEDS]
    accs = [a for a in accs if a is not None]
    if accs:
        print(f'  {approach:12s}: {np.mean(accs):.1f} \u00b1 {np.std(accs, ddof=1):.1f}%  seeds={[f"{a:.1f}" for a in accs]}')

d = [SEED42_RESULTS['weld']['direct']] + [weld_results['direct'].get(s) for s in SEEDS]
a = [SEED42_RESULTS['weld']['augmented']] + [weld_results['augmented'].get(s) for s in SEEDS]
paired = [(av - dv) for dv, av in zip(d, a) if dv is not None and av is not None]
if paired:
    print(f'  DELTA       : +{np.mean(paired):.1f} \u00b1 {np.std(paired, ddof=1):.1f}pp')

---
## Final Summary & Statistical Tests

In [ ]:
from scipy import stats

all_task_results = {
    'granulometry': granulometry_results,
    'steel_surface': steel_results,
    'uhcs': uhcs_results,
    'weld': weld_results,
}

print('=' * 70)
print('ABLATION 5: STATISTICAL SIGNIFICANCE — FINAL RESULTS')
print('=' * 70)

summary = {}
for task_name, results in all_task_results.items():
    print(f'\n{task_name.upper()}:')
    summary[task_name] = {}

    for approach in ['direct', 'augmented']:
        accs = [SEED42_RESULTS[task_name][approach]] + [results[approach].get(s) for s in SEEDS]
        accs = [a for a in accs if a is not None]
        if accs:
            mean = np.mean(accs)
            std = np.std(accs, ddof=1) if len(accs) > 1 else 0.0
            print(f'  {approach:12s}: {mean:.1f} \u00b1 {std:.1f}%  (runs: {[f"{a:.1f}" for a in accs]})')
            summary[task_name][approach] = {'mean': round(mean, 1), 'std': round(std, 1), 'runs': accs}

    # Paired difference
    d_accs = [SEED42_RESULTS[task_name]['direct']] + [results['direct'].get(s) for s in SEEDS]
    a_accs = [SEED42_RESULTS[task_name]['augmented']] + [results['augmented'].get(s) for s in SEEDS]
    paired = [(a - d) for d, a in zip(d_accs, a_accs) if d is not None and a is not None]

    if paired:
        delta_mean = np.mean(paired)
        delta_std = np.std(paired, ddof=1) if len(paired) > 1 else 0.0
        print(f'  {"DELTA":12s}: +{delta_mean:.1f} \u00b1 {delta_std:.1f}pp  (CoT-Aug minus Direct)')

        if len(paired) >= 3:
            t_stat, p_value = stats.ttest_1samp(paired, 0)
            sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
            print(f'  {"":12s}  p={p_value:.4f} ({sig}) — paired t-test')
            summary[task_name]['delta'] = {'mean': round(delta_mean, 1), 'std': round(delta_std, 1), 'p_value': round(p_value, 4)}

# Save summary
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\n\nSummary saved to {RESULTS_DIR / "summary.json"}')